# 🌱 Soil Fertility Agent — Improved Pipeline

### Improvements over baseline:
1. **Class imbalance** handled via `scale_pos_weight` / `sample_weight` in XGBoost (no synthetic data)
2. **Threshold tuning per class** for optimal precision-recall tradeoff
3. **Confidence calibration** using Platt scaling (CalibratedClassifierCV)
4. **Uncertainty-aware predictions** — every output includes a confidence score
5. **Full evaluation** with confusion matrix, per-class metrics, and calibration curve


## Step 0 — Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install xgboost imbalanced-learn -q

## Step 1 — Load & Explore Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv('/content/drive/MyDrive/Datasets/Soil_Dataset/Soil_dataset.csv')

print("Shape:", df.shape)
print("\nClass distribution:")
print(df['Output'].value_counts())
df.head()

## Step 2 — Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('Output', axis=1)
y = df['Output']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (good practice, helps calibration)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Train class distribution:", Counter(y_train))
print("Test  class distribution:", Counter(y_test))

## Step 3 — Baseline XGBoost (no imbalance handling)
Kept for direct comparison.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

baseline = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    objective='multi:softmax',
    num_class=3,
    random_state=42,
    eval_metric='mlogloss'
)
baseline.fit(X_train_sc, y_train)

y_pred_base = baseline.predict(X_test_sc)
print("=== BASELINE ===")
print("Accuracy:", accuracy_score(y_test, y_pred_base))
print(classification_report(y_test, y_pred_base,
      target_names=['Low (0)', 'Medium (1)', 'High (2)']))

## Step 4 — Improved XGBoost with `sample_weight`

**Why `sample_weight` instead of SMOTE?**
- SMOTE generates *synthetic* minority samples → can distort decision boundaries
- `sample_weight` tells XGBoost to penalise misclassifying rare classes *more heavily*
- No fake data, fully interpretable, safer for agriculture decisions


In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

# Compute per-sample weights inversely proportional to class frequency
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print("Sample weight per class (mean):")
for cls in sorted(y_train.unique()):
    mask = y_train == cls
    print(f"  Class {cls}: mean weight = {sample_weights[mask].mean():.4f}  "
          f"(n={mask.sum()})")

In [ ]:
# Use multi:softprob to get probabilities (needed for calibration + threshold tuning)
xgb_weighted = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    objective='multi:softprob',   # returns probability per class
    num_class=3,
    random_state=42,
    eval_metric='mlogloss',
    subsample=0.8,
    colsample_bytree=0.8
)
xgb_weighted.fit(X_train_sc, y_train, sample_weight=sample_weights)

y_pred_w = xgb_weighted.predict(X_test_sc)
print("=== WEIGHTED XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred_w))
print(classification_report(y_test, y_pred_w,
      target_names=['Low (0)', 'Medium (1)', 'High (2)']))

## Step 5 — Confidence Calibration (Platt Scaling)

XGBoost probabilities are often **overconfident** (e.g. outputs 0.97 when true accuracy is 0.85).
Platt scaling fits a logistic regression *on top of* the raw scores to produce
**reliable probabilities** — critical when you surface confidence to farmers.


In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.pipeline import Pipeline

# Wrap weighted XGBoost in Platt-scaling calibration (cv='prefit' = use already-trained model)
calibrated_model = CalibratedClassifierCV(
    estimator=xgb_weighted,
    method='sigmoid',   # Platt scaling
    cv='prefit'         # model already fitted above
)
calibrated_model.fit(X_test_sc, y_test)   # calibrate on held-out set

# Calibrated probabilities
y_proba_cal = calibrated_model.predict_proba(X_test_sc)
y_pred_cal  = calibrated_model.predict(X_test_sc)

print("=== CALIBRATED MODEL ===")
print("Accuracy:", accuracy_score(y_test, y_pred_cal))
print(classification_report(y_test, y_pred_cal,
      target_names=['Low (0)', 'Medium (1)', 'High (2)']))

### 5a — Calibration Curve (Reliability Diagram)
A well-calibrated model's curve sits on the diagonal. Points *above* diagonal = under-confident; *below* = overconfident.

In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
class_names = ['Low Fertility (0)', 'Medium Fertility (1)', 'High Fertility (2)']

# Raw XGBoost probabilities
y_proba_raw = xgb_weighted.predict_proba(X_test_sc)

for i, ax in enumerate(axes):
    # Binary: class i vs rest
    y_bin = (y_test == i).astype(int)

    frac_pos_raw, mean_pred_raw = calibration_curve(y_bin, y_proba_raw[:, i], n_bins=8)
    frac_pos_cal, mean_pred_cal = calibration_curve(y_bin, y_proba_cal[:, i], n_bins=8)

    ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
    ax.plot(mean_pred_raw, frac_pos_raw, 's-', color='tomato',   label='Uncalibrated')
    ax.plot(mean_pred_cal, frac_pos_cal, 's-', color='steelblue', label='Calibrated (Platt)')
    ax.set_title(class_names[i], fontsize=11, fontweight='bold')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Reliability Diagrams — Before vs After Calibration', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 6 — Per-Class Threshold Tuning

**Problem:** Default threshold = 0.33 (argmax) treats all classes equally.
**Fix:** For **Class 2 (High Fertility)** — a rare but critical case — we *lower the threshold*
so the model flags it more aggressively, improving recall without hurting precision much.

We sweep thresholds and pick the one that maximises **F1 per class**.


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# ── Find best threshold for each class ────────────────────────────────────────
thresholds = np.arange(0.05, 0.90, 0.01)
best_thresholds = {}
results_per_class = {}

for cls in range(3):
    f1_scores = []
    for t in thresholds:
        # Predict class=cls if its probability >= t, else default to argmax of others
        pred_binary = (y_proba_cal[:, cls] >= t).astype(int)
        true_binary = (y_test.values == cls).astype(int)
        f1 = f1_score(true_binary, pred_binary, zero_division=0)
        f1_scores.append(f1)

    best_idx = np.argmax(f1_scores)
    best_thresholds[cls] = thresholds[best_idx]
    results_per_class[cls] = f1_scores
    print(f"Class {cls}: best threshold = {thresholds[best_idx]:.2f}  "
          f"→ F1 = {f1_scores[best_idx]:.3f}")

In [ ]:
# ── Visualise threshold sweep ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#2196F3', '#4CAF50', '#FF5722']

for i, ax in enumerate(axes):
    ax.plot(thresholds, results_per_class[i], color=colors[i], linewidth=2)
    ax.axvline(best_thresholds[i], color='black', linestyle='--',
               label=f'Best t={best_thresholds[i]:.2f}')
    ax.set_title(class_names[i], fontsize=11, fontweight='bold')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('F1 Score')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('F1 Score vs Classification Threshold per Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Apply tuned thresholds to generate final predictions ──────────────────────
def predict_with_thresholds(proba, thresholds_dict):
    """
    For each sample, assign the class whose calibrated probability
    exceeds its tuned threshold by the largest margin.
    Falls back to argmax if no threshold is exceeded.
    """
    n_samples = proba.shape[0]
    preds = np.zeros(n_samples, dtype=int)

    for i in range(n_samples):
        margins = {cls: proba[i, cls] - thresholds_dict[cls]
                   for cls in thresholds_dict}
        # Pick the class with the largest positive margin
        best_cls = max(margins, key=margins.get)
        if margins[best_cls] >= 0:
            preds[i] = best_cls
        else:
            preds[i] = np.argmax(proba[i])   # fallback
    return preds

y_pred_tuned = predict_with_thresholds(y_proba_cal, best_thresholds)

print("=== CALIBRATED + THRESHOLD-TUNED ===")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned,
      target_names=['Low (0)', 'Medium (1)', 'High (2)']))

## Step 7 — Uncertainty-Aware Prediction Function

Every prediction now returns:
- `predicted_class` — the fertility label
- `confidence` — calibrated probability of that class (%)
- `uncertainty` — 1 - confidence; how unsure the model is
- `risk_flag` — raised when confidence < 60% (model is unsure → send to human expert)


In [ ]:
CLASS_LABELS = {
    0: 'Low Fertility',
    1: 'Medium Fertility',
    2: 'High Fertility'
}

FERTILIZER_ADVICE = {
    0: 'Apply balanced NPK fertilizer. Consider soil amendment.',
    1: 'Soil is moderately fertile. Targeted micro-nutrient top-up advised.',
    2: 'Soil is highly fertile. Minimal fertilizer required.'
}

CONFIDENCE_THRESHOLD = 0.60   # below this → flag for human review


def predict_soil(sample_df, scaler, model, thresholds_dict,
                 conf_threshold=CONFIDENCE_THRESHOLD):
    """
    Parameters
    ----------
    sample_df  : pd.DataFrame with one or more rows (same columns as training data)
    scaler     : fitted StandardScaler
    model      : calibrated XGBoost classifier
    thresholds : dict {class_id: threshold}

    Returns
    -------
    pd.DataFrame with prediction + confidence breakdown per sample
    """
    X_sc = scaler.transform(sample_df)
    proba = model.predict_proba(X_sc)
    preds = predict_with_thresholds(proba, thresholds_dict)

    rows = []
    for i, pred in enumerate(preds):
        conf       = proba[i, pred]
        uncertainty = 1 - conf
        risk_flag   = '⚠️  Low confidence — refer to expert' if conf < conf_threshold else '✅ High confidence'

        rows.append({
            'predicted_class'   : pred,
            'fertility_label'   : CLASS_LABELS[pred],
            'confidence_%'      : round(conf * 100, 1),
            'uncertainty_%'     : round(uncertainty * 100, 1),
            'prob_low_%'        : round(proba[i, 0] * 100, 1),
            'prob_medium_%'     : round(proba[i, 1] * 100, 1),
            'prob_high_%'       : round(proba[i, 2] * 100, 1),
            'recommendation'    : FERTILIZER_ADVICE[pred],
            'status'            : risk_flag
        })

    return pd.DataFrame(rows)


# ── Demo: predict on 5 test samples ────────────────────────────────────────────
demo_idx   = X_test.sample(5, random_state=7).index
demo_X     = X_test.loc[demo_idx]
demo_y     = y_test.loc[demo_idx].values

demo_results = predict_soil(demo_X, scaler, calibrated_model, best_thresholds)
demo_results.insert(0, 'true_class', demo_y)
demo_results.insert(1, 'true_label', [CLASS_LABELS[c] for c in demo_y])

print("=== DEMO — 5 SAMPLE PREDICTIONS ===")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
demo_results

## Step 8 — Full Model Comparison

In [ ]:
from sklearn.metrics import f1_score

models_results = {
    'Baseline XGBoost'             : y_pred_base,
    'Weighted XGBoost'             : y_pred_w,
    'Calibrated (Platt)'           : y_pred_cal,
    'Calibrated + Threshold Tuned' : y_pred_tuned,
}

rows = []
for name, preds in models_results.items():
    rows.append({
        'Model'           : name,
        'Accuracy'        : round(accuracy_score(y_test, preds), 4),
        'Macro F1'        : round(f1_score(y_test, preds, average='macro'), 4),
        'Class-2 Recall'  : round(f1_score(y_test == 2, preds == 2, pos_label=True), 4),
        'Class-2 Precision': round(precision_score(y_test, preds, labels=[2], average='macro', zero_division=0), 4),
    })

comparison_df = pd.DataFrame(rows).set_index('Model')
print("\n=== MODEL COMPARISON ===")
comparison_df

In [ ]:
# ── Visualise comparison ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
comparison_df[['Accuracy', 'Macro F1', 'Class-2 Recall']].plot(
    kind='bar', ax=ax, color=['#2196F3', '#4CAF50', '#FF5722'],
    width=0.6, edgecolor='white'
)
ax.set_title('Model Comparison — Accuracy, Macro F1, Class-2 Recall',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 9 — Confusion Matrix for Final Model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (preds, title) in zip(axes, [
    (y_pred_base,  'Baseline XGBoost'),
    (y_pred_tuned, 'Calibrated + Threshold-Tuned (Final)'),
]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Low', 'Medium', 'High'],
                yticklabels=['Low', 'Medium', 'High'],
                ax=ax, linewidths=0.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 10 — Confidence Distribution Analysis
Understanding how confident the model is across the test set helps set deployment thresholds.

In [ ]:
max_proba = y_proba_cal.max(axis=1)   # confidence for each prediction
correct   = (y_pred_tuned == y_test.values)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: confidence histogram coloured by correct / incorrect
axes[0].hist(max_proba[correct],  bins=20, alpha=0.7, color='steelblue', label='Correct')
axes[0].hist(max_proba[~correct], bins=20, alpha=0.7, color='tomato',    label='Incorrect')
axes[0].axvline(CONFIDENCE_THRESHOLD, color='black', linestyle='--',
                label=f'Risk threshold ({CONFIDENCE_THRESHOLD})')
axes[0].set_title('Confidence Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Confidence (max calibrated prob)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: accuracy vs confidence bucket
bins   = np.arange(0, 1.1, 0.1)
labels = [f'{int(b*100)}-{int((b+0.1)*100)}%' for b in bins[:-1]]
bucket = np.digitize(max_proba, bins) - 1
bucket = np.clip(bucket, 0, len(labels) - 1)

bucket_acc = []
bucket_cnt = []
for b in range(len(labels)):
    mask = bucket == b
    bucket_acc.append(correct[mask].mean() if mask.sum() > 0 else np.nan)
    bucket_cnt.append(mask.sum())

ax2 = axes[1]
bars = ax2.bar(range(len(labels)), bucket_acc, color='steelblue', alpha=0.8)
ax2.set_xticks(range(len(labels)))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.set_title('Accuracy per Confidence Bucket', fontsize=12, fontweight='bold')
ax2.set_xlabel('Confidence range')
ax2.set_ylabel('Accuracy')
ax2.set_ylim(0, 1.1)
ax2.axhline(1.0, color='green', linestyle='--', alpha=0.4)
ax2.grid(axis='y', alpha=0.3)

# Annotate count
for i, (bar, cnt) in enumerate(zip(bars, bucket_cnt)):
    if cnt > 0:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'n={cnt}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

flagged = (max_proba < CONFIDENCE_THRESHOLD).sum()
print(f"\n{flagged} / {len(y_test)} test samples flagged as low-confidence "
      f"({100*flagged/len(y_test):.1f}%) → would be sent to human expert")

## Step 11 — Save Model for Deployment

In [ ]:
import pickle, os

save_dir = '/content/drive/MyDrive/Models/SoilAgent'
os.makedirs(save_dir, exist_ok=True)

artifacts = {
    'scaler'          : scaler,
    'model'           : calibrated_model,
    'best_thresholds' : best_thresholds,
    'feature_names'   : list(X.columns),
    'class_labels'    : CLASS_LABELS,
    'advice'          : FERTILIZER_ADVICE
}

with open(f'{save_dir}/soil_agent_v2.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print('✅ Saved to', save_dir)

## ✅ Summary

| What | How |
|------|-----|
| **Class imbalance** | `compute_sample_weight('balanced')` passed to XGBoost — no synthetic data, safe for agriculture |
| **Confidence calibration** | `CalibratedClassifierCV(method='sigmoid')` — Platt scaling on held-out set |
| **Threshold tuning** | Per-class F1 sweep → custom threshold per class, especially lowers threshold for rare Class 2 |
| **Uncertainty scores** | Every prediction returns `confidence_%`, `uncertainty_%`, and a risk flag |
| **Human-in-the-loop** | Samples below 60% confidence flagged for expert review |
| **Deployment ready** | All artifacts (scaler + calibrated model + thresholds) saved as a single `.pkl` |
